# Get counts of significant associations

In [ ]:
import pandas as pd

In [ ]:
# both of these come from running the cleaning_and_high_level notebook
df = pd.read_csv('output/cleaned_data.csv', index_col=0)
factor_idx = pd.read_csv('output/factor_associations.csv', index_col=0).index

df.head()

,Short Name,Title,Ages,Genders,PA Type,PA Measurement,Factor,Factor Measurement,Tested,Positive,Negative,Notes,Continent,below_12,above_12,PA Measurement Category
0,Aibar 2015,"Effect of weather, school transport, and perce...",14.33 (0.73),All,MVPA,accelerometer,Poi Availability,Self-report,1,0,0,NaN,Europe,False,True,objective
1,Aibar 2015,"Effect of weather, school transport, and perce...",14.33 (0.73),All,MVPA,accelerometer,Street Connectivity,Self-report,1,0,0,NaN,Europe,False,True,objective
2,Aibar 2015,"Effect of weather, school transport, and perce...",14.33 (0.73),All,MVPA,accelerometer,Walking Amenities,Self-report,1,0,0,NaN,Europe,False,True,objective
3,Aibar 2015,"Effect of weather, school transport, and perce...",14.33 (0.73),All,MVPA,accelerometer,Aesthetics,Self-report,1,0,0,NaN,Europe,False,True,objective
4,Aibar 2015,"Effect of weather, school transport, and perce...",14.33 (0.73),All,MVPA,accelerometer,Traffic Safety,Self-report,1,0,0,NaN,Europe,False,True,objective


How to define the trends:
- If there are 15% significant associations: small trend
- If  more than 75% of significant associations are in one direction: it goes in that direction

In [ ]:
mid_cutoff = 15
small_cutoff = 100 # don't want to include this for now

def get_trend(s):
    # if none were tested
    if s['Tested'] < 5:
        return '?'
    
    # check if there were any significant results 
    num_signif = s['Positive'] + s['Negative']
    if num_signif == 0:
        return '0'
    
    # get percent significant in each direction
    perc_signif = 100 * num_signif / s['Tested']
    perc_pos = 100 * s['Positive'] / num_signif
    perc_neg = 100 * s['Negative'] / num_signif

    if ((perc_signif >= mid_cutoff) and (perc_pos < 75) and (perc_neg) < 75):       # moderate mixed trend
        return '+-'    
    elif ((perc_signif >= small_cutoff) and (perc_pos < 75) and (perc_neg) < 75):   # small mixed trend
        return '(+-)'
    elif s['Percent Positive'] >= mid_cutoff:                                       # moderate positive trend
        return '+'
    elif s['Percent Positive'] >= small_cutoff:                                     # small positive trend
        return '(+)'
    elif s['Percent Negative'] >= mid_cutoff:                                       # moderate negative trend
        return '-'
    elif s['Percent Negative'] >= small_cutoff:                                     # small negative trend
        return '(-)'
    else:
        return '0'


def get_overall(df, associations):
    '''Get the overall trend with cul-de-sacs reverse coded'''
    cul_de_sac = associations.loc[['Cul-De-Sacs']]
    reverse_cul_de_sac = cul_de_sac.copy()

    reverse_cul_de_sac['Positive'] = cul_de_sac['Negative']
    reverse_cul_de_sac['Negative'] = cul_de_sac['Positive']

    reverse_cul_de_sac.index = pd.Index(data=['Reverse Cul-De-Sacs'], name='Factor')


    summary_associations = pd.concat([associations, reverse_cul_de_sac])
    summary_associations = summary_associations.drop('Cul-De-Sacs')

    summary_dict = {'Tested': summary_associations['Tested'].sum(), 
                    'Positive': summary_associations['Positive'].sum(), 
                    'Negative': summary_associations['Negative'].sum(),
                    'Number of Studies': df['Short Name'].nunique()}

    summary = pd.DataFrame(summary_dict, index=pd.Index(data=['Overall'], name='Factor'))

    return pd.concat([associations, summary])


def get_associations(df):
    '''Get the associations for the dataframe and format for the paper'''
    associations = df.groupby('Factor')[['Tested', 'Positive', 'Negative']].sum()
    associations = associations.reindex(factor_idx, fill_value=0) # make sure we capture when a factor was not tested

    # get the number of studies for each factor
    factor_studies = df[['Short Name', 'Factor']].drop_duplicates()
    factor_studies = factor_studies['Factor'].value_counts()
    factor_studies = factor_studies.reindex(factor_idx, fill_value=0)   # make sure we capture when a factor was not tested
    associations = associations.merge(pd.DataFrame({'Number of Studies': factor_studies}), left_index=True, right_index=True)

    associations = get_overall(df, associations)
    associations['Percent Positive'] = 100 * associations['Positive'] / associations['Tested']
    associations['Percent Negative'] = 100 * associations['Negative'] / associations['Tested']
    associations['Null'] = associations['Tested'] - associations['Positive'] - associations['Negative']
    associations['Trend'] = associations.apply(get_trend, axis=1)

    # round for presentation
    associations['Percent Positive'] = associations['Percent Positive'].round(2)
    associations['Percent Negative'] = associations['Percent Negative'].round(2)

    # get the number of studies
    factor_studies = df[['Short Name', 'Factor']].drop_duplicates()
    factor_studies = factor_studies['Factor'].value_counts()

    return associations


def format_full(df):
    '''Get full version of association table'''
    return df[['Number of Studies', 'Tested', 'Positive', 'Null', 'Negative', 'Percent Positive', 'Percent Negative', 'Trend']]


def format_short(df):
    '''Make a smaller version of the associations table for readability'''
    df = df.rename(columns={'Tested': 'N'})

    return df[['Number of Studies', 'N', 'Percent Positive', 'Percent Negative', 'Trend']]


def format_shorter(df):
    '''Make an even smaller version of associations table'''
    df["# Studies (# Assn.)"] = [f'{a} ({b})' for a, b in zip(df['Number of Studies'], df['Tested'])]
    df = df.rename(columns={'Percent Positive': 'Positive (%)', 'Percent Negative': 'Negative (%)'})

    return df[['# Studies (# Assn.)', 'Positive (%)', 'Negative (%)', 'Trend']]


### Overall Associations

In [ ]:
overall = get_associations(df)
format_full(overall).to_csv('output/overall_associations.csv')

format_full(overall)

,Number of Studies,Tested,Positive,Null,Negative,Percent Positive,Percent Negative,Trend
Factor,,,,,,,,
Walkability Index,38,331,27,269,35,8.16,10.57,+-
Recreation Availability,31,288,49,227,12,17.01,4.17,+
Traffic Safety,25,247,25,202,20,10.12,8.10,+-
Aesthetics,23,164,29,134,1,17.68,0.61,+
Poi Availability,22,163,13,136,14,7.98,8.59,+-
Walking Amenities,22,166,20,136,10,12.05,6.02,+-
Residential Density,22,94,14,74,6,14.89,6.38,+-
Street Connectivity,21,114,5,99,10,4.39,8.77,0
Crime Safety,16,84,12,70,2,14.29,2.38,0


### Associations by Age Range

In [ ]:
b12_associations = get_associations(df[df['below_12']])
a12_associations = get_associations(df[df['above_12']])
below_12 = format_shorter(b12_associations)
above_12 = format_shorter(a12_associations)

cols = below_12.columns

below_12.columns = pd.MultiIndex.from_tuples([('Below 12', col) for col in cols])
above_12.columns= pd.MultiIndex.from_tuples([('Above 12', col) for col in cols])

age_formatted = below_12.merge(above_12, left_index=True, right_index=True)
age_formatted.to_csv('output/age_associations.csv')

age_formatted.loc[factor_idx]

Below 12                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                  10 (115)         1.74        14.78     0   
Recreation Availability            12 (112)        20.54         5.36     +   
Traffic Safety                     10 (131)        12.21         7.63    +-   
Aesthetics                           8 (73)        31.51         0.00     +   
Poi Availability                     7 (70)         7.14        20.00    +-   
Walking Amenities                    9 (78)        10.26         8.97    +-   
Residential Density                  5 (24)        12.50        12.50    +-   
Street Connectivity                  5 (22)        13.64        18.18    +-   
Crime Safety                         4 (31)         9.68         3.23     0   
Accessibility                        4 (31)        32.26         0.00     +   
Land Use Diversity                    3 (9)         0.00        33.33     -   
Cul-De-Sacs                          3 (12)        25.00         0.00     +   
Transit Density                      5 (31)        12.90        12.90    +-   

                                   Above 12                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                  17 (112)        15.18         8.04    +-  
Recreation Availability            12 (106)        17.92         0.00     +  
Traffic Safety                      12 (90)         4.44        11.11    +-  
Aesthetics                          12 (76)         3.95         1.32     0  
Poi Availability                    12 (81)         9.88         0.00     0  
Walking Amenities                   10 (70)        12.86         4.29     0  
Residential Density                 11 (48)        12.50         0.00     0  
Street Connectivity                  9 (73)         2.74         4.11     0  
Crime Safety                         7 (40)        15.00         0.00     +  
Accessibility                        6 (32)        15.62         0.00     +  
Land Use Diversity                   5 (12)         8.33        25.00     -  
Cul-De-Sacs                           2 (6)        83.33         0.00     +  
Transit Density                       3 (8)         0.00         0.00     0

In [ ]:
b12_sum = b12_associations.sum(axis=0)
a12_sum = a12_associations.sum(axis=0)

b12_sum['Percent Significant'] = (b12_sum['Positive'] + b12_sum['Negative']) / b12_sum['Tested']
a12_sum['Percent Significant'] = (a12_sum['Positive'] + a12_sum['Negative']) / a12_sum['Tested']

print('Percent Significant Below 12:', b12_sum['Percent Significant'])
print('Percent Significant Above 12:', a12_sum['Percent Significant'])

Percent Significant Below 12: 0.2327469553450609
Percent Significant Above 12: 0.15119363395225463


### Associations by PA report type

In [ ]:
objective = format_shorter(get_associations(df[df['PA Measurement Category'] == 'objective']))
self_report_pa = format_shorter(get_associations(df[df['PA Measurement Category'] == 'self-report']))
parent_report_pa = format_shorter(get_associations(df[df['PA Measurement Category'] == 'parent-report']))

objective.columns = pd.MultiIndex.from_tuples([('Objective', col) for col in cols])
self_report_pa.columns = pd.MultiIndex.from_tuples([('Self-report', col) for col in cols])
parent_report_pa.columns = pd.MultiIndex.from_tuples([('Parent-report', col) for col in cols])

pa_report = objective.merge(self_report_pa, left_index=True, right_index=True)
pa_report = pa_report.merge(parent_report_pa, left_index=True, right_index=True)

pa_report.to_csv('output/pa_report_associations.csv')
pa_report

Objective                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                  22 (113)        15.04         8.85    +-   
Recreation Availability            19 (125)        14.40         5.60    +-   
Traffic Safety                     16 (111)         9.91         7.21    +-   
Aesthetics                          13 (71)         8.45         0.00     0   
Poi Availability                    12 (58)         6.90         3.45     0   
Walking Amenities                   13 (75)        10.67         5.33    +-   
Residential Density                 12 (30)        20.00         3.33     +   
Street Connectivity                 14 (33)         6.06         3.03     0   
Crime Safety                         8 (12)         8.33         8.33    +-   
Accessibility                        9 (29)        17.24         0.00     +   
Land Use Diversity                   7 (19)         5.26         5.26     0   
Cul-De-Sacs                          5 (15)        13.33         6.67    +-   
Transit Density                      5 (18)        11.11         0.00     0   
Overall                            34 (709)        11.57         5.22    +-   

                                Self-report                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                  20 (190)         5.26         8.42     0   
Recreation Availability            13 (110)        12.73         2.73     0   
Traffic Safety                      10 (56)         3.57        17.86     -   
Aesthetics                          13 (55)         9.09         1.82     0   
Poi Availability                    10 (60)        10.00         0.00     0   
Walking Amenities                   10 (46)        10.87         6.52    +-   
Residential Density                 11 (38)        13.16         5.26    +-   
Street Connectivity                  6 (60)         0.00         5.00     0   
Crime Safety                         9 (53)        15.09         1.89     +   
Accessibility                        7 (24)        16.67         0.00     +   
Land Use Diversity                    2 (4)         0.00        75.00     ?   
Cul-De-Sacs                           4 (7)        42.86        42.86    +-   
Transit Density                      4 (21)         9.52         0.00     0   
Overall                            29 (724)         8.84         6.22    +-   

                              Parent-report                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                    4 (25)         0.00        36.00     -  
Recreation Availability             10 (50)        34.00         4.00     +  
Traffic Safety                      10 (80)        15.00         2.50     +  
Aesthetics                           7 (35)        48.57         0.00     +  
Poi Availability                     7 (42)         7.14        28.57     -  
Walking Amenities                    9 (45)        15.56         6.67    +-  
Residential Density                  6 (26)        11.54        11.54    +-  
Street Connectivity                  5 (21)        14.29        28.57    +-  
Crime Safety                         4 (19)        15.79         0.00     +  
Accessibility                        5 (28)        46.43         0.00     +  
Land Use Diversity                    1 (1)         0.00       100.00     ?  
Cul-De-Sacs                           2 (3)       100.00         0.00     ?  
Transit Density                       3 (4)         0.00       100.00     ?  
Overall                            12 (379)        20.58        11.87    +-

### Associations by walkability report type

In [ ]:
gis = format_shorter(get_associations(df[df['Factor Measurement'] == 'Gis']))
audit = format_shorter(get_associations(df[df['Factor Measurement'] == 'Audit']))
self_report_walk = format_shorter(get_associations(df[df['Factor Measurement'] == 'Self-report']))
parent_report_walk = format_shorter(get_associations(df[df['Factor Measurement'] == 'Parent-report']))

gis.columns = pd.MultiIndex.from_tuples([('GIS', col) for col in cols])
audit.columns = pd.MultiIndex.from_tuples([('Audit', col) for col in cols])
self_report_walk.columns = pd.MultiIndex.from_tuples([('Self-report', col) for col in cols])
parent_report_walk.columns = pd.MultiIndex.from_tuples([('Parent-report', col) for col in cols])

gis_audit = gis.merge(audit, left_index=True, right_index=True)
gis_audit.to_csv('output/gis_audit_associations.csv')
gis_audit

GIS                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                  32 (290)         8.62        10.34    +-   
Recreation Availability            18 (158)        12.03         6.33    +-   
Traffic Safety                       2 (14)         0.00         0.00     0   
Aesthetics                            0 (0)          NaN          NaN     ?   
Poi Availability                      3 (9)         0.00         0.00     0   
Walking Amenities                     1 (2)         0.00         0.00     ?   
Residential Density                 10 (40)        17.50         7.50    +-   
Street Connectivity                 10 (59)         1.69         3.39     0   
Crime Safety                         1 (18)        11.11         5.56    +-   
Accessibility                         0 (0)          NaN          NaN     ?   
Land Use Diversity                   7 (19)         5.26        21.05     -   
Cul-De-Sacs                           3 (9)         0.00        33.33     -   
Transit Density                      3 (24)        16.67         0.00     +   
Overall                            34 (642)         9.66         7.79    +-   

                                      Audit                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                    3 (18)         0.00         0.00     0  
Recreation Availability              6 (72)         8.33         0.00     0  
Traffic Safety                      6 (153)         7.84        13.07    +-  
Aesthetics                           6 (94)        19.15         1.06     +  
Poi Availability                     6 (86)         4.65        16.28     -  
Walking Amenities                   6 (100)         9.00        10.00    +-  
Residential Density                  2 (14)         0.00        21.43     -  
Street Connectivity                   0 (0)          NaN          NaN     ?  
Crime Safety                          0 (0)          NaN          NaN     ?  
Accessibility                        1 (20)        25.00         0.00     +  
Land Use Diversity                    2 (8)         0.00        25.00     -  
Cul-De-Sacs                          2 (14)        57.14         0.00     +  
Transit Density                      3 (19)         0.00        21.05     -  
Overall                             8 (598)         9.03        10.37    +-

In [ ]:
self_parent = self_report_pa.merge(parent_report_pa, left_index=True, right_index=True)
self_parent.to_csv('output/self_parent_report_associations.csv')

self_parent

Self-report                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                  20 (190)         5.26         8.42     0   
Recreation Availability            13 (110)        12.73         2.73     0   
Traffic Safety                      10 (56)         3.57        17.86     -   
Aesthetics                          13 (55)         9.09         1.82     0   
Poi Availability                    10 (60)        10.00         0.00     0   
Walking Amenities                   10 (46)        10.87         6.52    +-   
Residential Density                 11 (38)        13.16         5.26    +-   
Street Connectivity                  6 (60)         0.00         5.00     0   
Crime Safety                         9 (53)        15.09         1.89     +   
Accessibility                        7 (24)        16.67         0.00     +   
Land Use Diversity                    2 (4)         0.00        75.00     ?   
Cul-De-Sacs                           4 (7)        42.86        42.86    +-   
Transit Density                      4 (21)         9.52         0.00     0   
Overall                            29 (724)         8.84         6.22    +-   

                              Parent-report                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                    4 (25)         0.00        36.00     -  
Recreation Availability             10 (50)        34.00         4.00     +  
Traffic Safety                      10 (80)        15.00         2.50     +  
Aesthetics                           7 (35)        48.57         0.00     +  
Poi Availability                     7 (42)         7.14        28.57     -  
Walking Amenities                    9 (45)        15.56         6.67    +-  
Residential Density                  6 (26)        11.54        11.54    +-  
Street Connectivity                  5 (21)        14.29        28.57    +-  
Crime Safety                         4 (19)        15.79         0.00     +  
Accessibility                        5 (28)        46.43         0.00     +  
Land Use Diversity                    1 (1)         0.00       100.00     ?  
Cul-De-Sacs                           2 (3)       100.00         0.00     ?  
Transit Density                       3 (4)         0.00       100.00     ?  
Overall                            12 (379)        20.58        11.87    +-

Interesting finding that in self-report, there are 0 negative associations

### Associations by gender

In [ ]:
girls_associations = get_associations(df[df['Genders'] == 'Girls'])
boys_associations = get_associations(df[df['Genders'] == 'Boys'])

girls = format_shorter(girls_associations)
boys = format_shorter(boys_associations)

girls.columns = pd.MultiIndex.from_tuples([('Girls', col) for col in cols])
boys.columns = pd.MultiIndex.from_tuples([('Boys', col) for col in cols])

gender = girls.merge(boys, left_index=True, right_index=True)
gender.to_csv('output/gender_associations.csv')

gender

Girls                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                    9 (32)         9.38         0.00     0   
Recreation Availability              8 (46)         8.70         0.00     0   
Traffic Safety                       5 (19)         5.26         0.00     0   
Aesthetics                           4 (14)         7.14         0.00     0   
Poi Availability                     3 (17)        11.76         0.00     0   
Walking Amenities                     2 (8)         0.00         0.00     0   
Residential Density                  4 (11)        27.27         0.00     +   
Street Connectivity                  4 (29)         0.00         3.45     0   
Crime Safety                         4 (19)        10.53         0.00     0   
Accessibility                         2 (3)        66.67         0.00     ?   
Land Use Diversity                    3 (6)         0.00        16.67     -   
Cul-De-Sacs                           0 (0)          NaN          NaN     ?   
Transit Density                      2 (10)         0.00         0.00     0   
Overall                            12 (214)         8.41         0.93     0   

                                       Boys                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                    7 (27)         0.00         0.00     0  
Recreation Availability              6 (40)        15.00         0.00     +  
Traffic Safety                       5 (19)        15.79         0.00     +  
Aesthetics                           3 (11)         0.00         0.00     0  
Poi Availability                     2 (14)        14.29         0.00     0  
Walking Amenities                     2 (8)        37.50         0.00     +  
Residential Density                  4 (11)         0.00         9.09     0  
Street Connectivity                  4 (29)         0.00         0.00     0  
Crime Safety                         4 (21)        19.05         4.76     +  
Accessibility                         1 (2)         0.00         0.00     ?  
Land Use Diversity                    2 (3)         0.00         0.00     ?  
Cul-De-Sacs                           0 (0)          NaN          NaN     ?  
Transit Density                      2 (11)        18.18         0.00     +  
Overall                            10 (196)        10.20         1.02     0

### Associations by region

In [ ]:
na_associations = get_associations(df[df['Continent'] == 'North America'])
europe_associations = get_associations(df[df['Continent'] == 'Europe'])
australia_associations = get_associations(df[(df['Continent'] == 'Australia') | (df['Continent'] == 'New Zealand')])

na = format_shorter(na_associations)
europe = format_shorter(europe_associations)
australia = format_shorter(australia_associations)

na.columns = pd.MultiIndex.from_tuples([('North America', col) for col in cols])
europe.columns  = pd.MultiIndex.from_tuples([('Europe', col) for col in cols])
australia.columns = pd.MultiIndex.from_tuples([('Australia / NZ', col) for col in cols])

region = na.merge(europe, left_index=True, right_index=True)
region = region.merge(australia, left_index=True, right_index=True)
region.to_csv('output/region_associations.csv')

region

North America                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                  21 (155)         4.52        18.71     -   
Recreation Availability            18 (143)        19.58         5.59     +   
Traffic Safety                     13 (187)        11.23         7.49    +-   
Aesthetics                         11 (116)        19.83         0.86     +   
Poi Availability                   11 (106)         5.66        11.32    +-   
Walking Amenities                  10 (112)        13.39         5.36    +-   
Residential Density                  9 (41)         4.88        12.20    +-   
Street Connectivity                  9 (34)         8.82        23.53    +-   
Crime Safety                         6 (50)        14.00         2.00     0   
Accessibility                        6 (55)        27.27         0.00     +   
Land Use Diversity                   5 (14)         7.14        35.71     -   
Cul-De-Sacs                          5 (20)        40.00        20.00    +-   
Transit Density                      4 (36)         5.56        11.11    +-   
Overall                           30 (1069)        12.54         9.45    +-   

                                     Europe                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                   8 (112)         9.82         0.89     0   
Recreation Availability              5 (52)        17.31         0.00     +   
Traffic Safety                       4 (12)        16.67         0.00     +   
Aesthetics                           4 (11)         0.00         0.00     0   
Poi Availability                     4 (11)         0.00         0.00     0   
Walking Amenities                    4 (15)        20.00         0.00     +   
Residential Density                  4 (15)        26.67         6.67     +   
Street Connectivity                  5 (16)         6.25        12.50    +-   
Crime Safety                          3 (9)        22.22         0.00     +   
Accessibility                         2 (9)        44.44         0.00     +   
Land Use Diversity                    2 (7)         0.00        14.29     0   
Cul-De-Sacs                           0 (0)          NaN          NaN     ?   
Transit Density                       2 (7)        28.57         0.00     +   
Overall                            12 (276)        13.77         1.81     0   

                             Australia / NZ                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                    7 (46)        15.22        10.87    +-  
Recreation Availability              6 (44)        13.64         9.09    +-  
Traffic Safety                       3 (26)         3.85        23.08     -  
Aesthetics                           2 (14)        35.71         0.00     +  
Poi Availability                     2 (13)         7.69        15.38    +-  
Walking Amenities                    2 (14)         0.00        28.57     -  
Residential Density                  3 (15)        20.00         0.00     +  
Street Connectivity                  3 (13)         7.69         0.00     0  
Crime Safety                          1 (2)         0.00         0.00     ?  
Accessibility                         2 (5)        20.00         0.00     +  
Land Use Diversity                    2 (6)         0.00         0.00     0  
Cul-De-Sacs                           1 (4)         0.00         0.00     ?  
Transit Density                       0 (0)          NaN          NaN     ?  
Overall                             8 (202)        12.38        10.40    +-

### Associations by PA Type

In [ ]:
mvpa_associations = get_associations(df[df['PA Type'] == 'MVPA'])
other_pa_associations = get_associations(df[~(df['PA Type'] == 'MVPA')])

mvpa = format_shorter(mvpa_associations)
other_pa = format_shorter(other_pa_associations)

mvpa.columns = pd.MultiIndex.from_tuples([('MVPA', col) for col in cols])
other_pa.columns = pd.MultiIndex.from_tuples([('Other PA', col) for col in cols])

pa_types = mvpa.merge(other_pa, left_index=True, right_index=True)
pa_types.to_csv('output/pa_type_association.csv')

pa_types

MVPA                                  \
                        # Studies (# Assn.) Positive (%) Negative (%) Trend   
Factor                                                                        
Walkability Index                   26 (87)        17.24         6.90    +-   
Recreation Availability             20 (96)        17.71         3.12     +   
Traffic Safety                      15 (72)         5.56         6.94     0   
Aesthetics                          12 (49)         8.16         0.00     0   
Poi Availability                    14 (50)         6.00         2.00     0   
Walking Amenities                   12 (52)         9.62         3.85     0   
Residential Density                 13 (27)        22.22         0.00     +   
Street Connectivity                 13 (23)         8.70         8.70    +-   
Crime Safety                         7 (10)        10.00        10.00    +-   
Accessibility                        8 (23)         8.70         0.00     0   
Land Use Diversity                   8 (18)         5.56        22.22     -   
Cul-De-Sacs                           4 (8)        25.00        12.50    +-   
Transit Density                      5 (12)        16.67         0.00     +   
Overall                            36 (527)        11.95         4.93    +-   

                                   Other PA                                  
                        # Studies (# Assn.) Positive (%) Negative (%) Trend  
Factor                                                                       
Walkability Index                  27 (244)         4.92        11.89    +-  
Recreation Availability            22 (192)        16.67         4.69     +  
Traffic Safety                     20 (175)        12.00         8.57    +-  
Aesthetics                         19 (115)        21.74         0.87     +  
Poi Availability                   15 (113)         8.85        11.50    +-  
Walking Amenities                  18 (114)        13.16         7.02    +-  
Residential Density                 14 (67)        11.94         8.96    +-  
Street Connectivity                 12 (91)         3.30         8.79     0  
Crime Safety                        13 (74)        14.86         1.35     0  
Accessibility                       11 (58)        34.48         0.00     +  
Land Use Diversity                    3 (9)         0.00        22.22     -  
Cul-De-Sacs                          6 (17)        35.29        17.65    +-  
Transit Density                      5 (31)         6.45        12.90    +-  
Overall                           41 (1300)        12.46         7.85    +-

In [ ]:
mvpa_sum = mvpa_associations.sum(axis=0)
other_pa_sum = other_pa_associations.sum(axis=0)

mvpa_sum['Percent Significant'] = (mvpa_sum['Positive'] + mvpa_sum['Negative']) / mvpa_sum['Tested']
other_pa_sum['Percent Significant'] = (other_pa_sum['Positive'] + other_pa_sum['Negative']) / other_pa_sum['Tested']
mvpa_sum['Percent Positive'] = (mvpa_sum['Positive']) / mvpa_sum['Tested']
other_pa_sum['Percent Positive'] = (other_pa_sum['Positive']) / other_pa_sum['Tested']
mvpa_sum['Percent Negative'] = (mvpa_sum['Negative']) / mvpa_sum['Tested']
other_pa_sum['Percent Negative'] = (other_pa_sum['Negative']) / other_pa_sum['Tested']

print('Percent Significant MVPA:', mvpa_sum['Percent Significant'])
print('Percent Significant other PA:', other_pa_sum['Percent Significant'])
print('Percent Positive MVPA:', mvpa_sum['Percent Positive'])
print('Percent Positive other PA:', other_pa_sum['Percent Positive'])
print('Percent Negative MVPA:', mvpa_sum['Percent Negative'])
print('Percent Negative other PA:', other_pa_sum['Percent Negative'])

Percent Significant MVPA: 0.16888045540796964
Percent Significant other PA: 0.20307692307692307
Percent Positive MVPA: 0.1204933586337761
Percent Positive other PA: 0.12576923076923077
Percent Negative MVPA: 0.04838709677419355
Percent Negative other PA: 0.07730769230769231


## Get the study names for each of the factors

In [ ]:
def format_with_names(df):
    '''Get names of each study for each factor from the ORIGINAL dataframe'''
    assn = get_associations(df)
    grouped_indices = df.groupby("Factor").indices

    name_dict = dict()

    for key, arr in grouped_indices.items():
        name_list = [df.iloc[i]['Short Name'] for i in arr]
        name_list = list(set(name_list))
        name_str = ', '.join(name_list)

        name_dict[key] = name_str
    
    name_df = pd.DataFrame.from_dict(name_dict, orient='index', columns=['Studies'])
    assn = assn.merge(name_df, left_index=True, right_index=True)

    return assn[['Studies', 'Tested', 'Percent Positive', 'Percent Negative', 'Trend']]

In [ ]:
format_with_names(df).to_csv("output/overall_names.csv")